# EUROSTAT

This notebook mirrors the EUROSTAT parser guide.


## Main entry point

Use `mario.parse_eurostat(...)` for both `SUT` and `IOT` workflows against the official Eurostat SDMX API.


## Key arguments

The key public arguments are:

- `country`: Eurostat geo code such as `IT` or `DE`
- `year`: reference year to parse
- `table`: choose `"SUT"` or `"IOT"`
- `iot_mode`: IOT-only selector, either `"product"` or `"industry"`
- `unit`: choose `"MIO_EUR"` or `"MIO_NAC"`
- `path`: optional local cache directory for raw SDMX CSV files
- `download`: when `True`, MARIO stores the raw CSV locally before parsing it


## Available years

Year availability depends on the country and on the table family.

- Official Eurostat metadata states that national `SUT` annual tables are published from `2010` onward.
- Official Eurostat metadata also states that national `IOT` tables are published from `2010` onward, but mandatory transmission is only for years ending in `0` or `5`.
- Additional `IOT` years depend on voluntary country transmission.

On the live official API, checked on **18 April 2026** with **Italy (`IT`)** as one concrete example, MARIO currently finds:

- `SUT`: `2010` to `2022`
- `IOT`: `2010` and then `2015` to `2022`

So for `SUT` you should expect a broadly annual series, while for `IOT` you should treat `2010`, `2015`, and `2020` as the core mandatory years and any extra years as country-specific.


## Relevant source pages and local cache

Relevant official pages:

- [ESA supply, use and input-output tables](https://ec.europa.eu/eurostat/web/esa-supply-use-input-tables)
- [Eurostat metadata for `naio_10_n`](https://ec.europa.eu/eurostat/cache/metadata/en/naio_10_n_esms.htm)
- [Eurostat API introduction](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-introduction)

`path` is optional and, when provided, acts as a raw SDMX cache:

```text
EUROSTAT/
└── cache/
    ├── NAIO_10_CP15_*.csv
    ├── NAIO_10_CP16_*.csv
    ├── NAIO_10_CP1700_*.csv
    └── NAIO_10_CP1750_*.csv
```

If `download=False`, MARIO reads directly from the API. If `download=True`, MARIO stores the raw slices under `path` before building the database.

In [1]:
import mario

## Parse a SUT directly from the API


In [2]:
db = mario.parse_eurostat(
    country="IT",
    year=2022,
    table="SUT",
)

db

INFO Parser: requesting Eurostat NAIO_10_CP15 for IT 2022 MIO_EUR.


INFO Parser: requesting Eurostat NAIO_10_CP16 for IT 2022 MIO_EUR.


INFO Parser: Eurostat SUT payload ready with shapes S=(65, 65), U=(65, 65), Yc=(65, 3), Va=(6, 65), Vc=(6, 65).


INFO Metadata: initialized.


name = Eurostat SUT IT 2022
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 65
Commodity = 65
Factor of production = 6
Satellite account = 1
Consumption category = 3
Region = 1

## Parse an IOT


In [3]:
db = mario.parse_eurostat(
    country="IT",
    year=2022,
    table="IOT",
    iot_mode="product",
)

db

INFO Parser: requesting Eurostat NAIO_10_CP1700 for IT 2022 MIO_EUR.


INFO Parser: Eurostat IOT payload ready with shapes Z=(65, 65), Y=(65, 3), V=(6, 65), mode=product.


INFO Metadata: initialized.


name = Eurostat IOT IT 2022 product
table = IOT
scenarios = ['baseline']
Factor of production = 6
Satellite account = 1
Consumption category = 3
Region = 1
Sector = 65

## Optional raw download and cache


In [4]:
mario.download_eurostat(
    path="/path/to/SUT",
    country="IT",
    year=2022,
    table="SUT",
)

{'table': 'SUT',
 'country': 'IT',
 'year': 2022,
 'unit': 'MIO_EUR',
 'iot_mode': None,
 'files': {'supply': '/path/to/eurostat_cache/NAIO_10_CP15_IT_2022_MIO_EUR.csv',
  'use': '/path/to/eurostat_cache/NAIO_10_CP16_IT_2022_MIO_EUR.csv'}}